# Building a Policy-Governed Agent with LangChain + AgentMesh
**Agent Governance Toolkit — Interactive Demo**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microsoft/agent-governance-toolkit/blob/main/notebooks/04_langchain_agentmesh_chatbot.ipynb)

In this notebook you will:
- Create a cryptographic identity for a customer support agent
- Define LangChain tools and wrap them with trust-based access control
- Add rate limiting to prevent runaway tool usage
- Build a real LangChain agent (with OpenAI) that enforces governance policies
- Inspect the audit trail to see what was allowed and blocked

**Scenario:** A customer support agent that can look up orders, check inventory, and update shipping — but is blocked from deleting customer accounts (trust too low) and processing refunds (missing capability), and is rate-limited to prevent abuse.

**Requires:** An OpenAI API key ([platform.openai.com/api-keys](https://platform.openai.com/api-keys)). This notebook uses `gpt-4o-mini` and costs less than $0.01 to run.

## Step 1 — Install and configure

In [ ]:
!pip install langchain langchain-openai langchain-agentmesh agent-os-kernel -q

In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

print("OPENAI_API_KEY is ready.")

## Step 2 — Create the chatbot's identity

Every agent needs a cryptographic identity — think of it as a passport. It contains a **DID** (unique ID), an **Ed25519 key pair** (for signing), and a list of **capabilities** (what this agent is allowed to do).

We also create an **agent card** — a signed credential the chatbot presents to prove who it is. We set the trust score to **0.7** (not the maximum) so we can demonstrate trust-score-based blocking later.

In [ ]:
from langchain_agentmesh import VerificationIdentity, TrustedAgentCard

bot_identity = VerificationIdentity.generate(
    "support-chatbot",
    capabilities=["lookup_order", "check_inventory", "cancel_account", "update_shipping"],
)
# The executor checks the *card's* capabilities and trust score, not the identity's.
bot_card = TrustedAgentCard(
    name="Support Chatbot",
    description="Handles order lookups, inventory checks, shipping updates, and account cancellations",
    capabilities=["lookup_order", "check_inventory", "cancel_account", "update_shipping"],
    trust_score=0.7,
)
bot_card.sign(bot_identity)

print(f"DID:          {bot_identity.did}")
print(f"Capabilities: {bot_identity.capabilities}")
print(f"Trust score:  {bot_card.trust_score}")
print(f"Card signed:  {bot_card.verify_signature()}")

## Step 3 — Define tools and wrap with trust gates

Five customer support tools, each requiring a different capability and trust level. The chatbot has capabilities for four of them but **not** `process_refund`. Its trust score is **0.7**, which is high enough for most tools but not for `cancel_account` (0.95).

| Tool | What it does | Required capability | Min trust | Expected result |
|------|-------------|--------------------:|----------:|-----------------|
| `lookup_order` | Look up a customer's order | `lookup_order` | 0.5 | Allowed |
| `check_inventory` | Check product stock | `check_inventory` | 0.5 | Allowed |
| `update_shipping` | Change shipping address | `update_shipping` | 0.6 | Allowed |
| `cancel_account` | Delete a customer account | `cancel_account` | 0.95 | **Blocked** (trust too low) |
| `process_refund` | Issue a refund | `process_refund` | 0.5 | **Blocked** (missing capability) |

In [ ]:
from langchain_core.tools import tool
from langchain_agentmesh import TrustGatedTool


@tool
def lookup_order(order_id: str) -> str:
    """Look up a customer order by ID to check its status and shipping info."""
    return f"Order {order_id}: Blue Backpack, shipped 2026-03-28, arrives 2026-04-02"


@tool
def check_inventory(product: str) -> str:
    """Check if a product is currently in stock."""
    return f"'{product}' is in stock — 42 units available"


@tool
def update_shipping(order_and_address: str) -> str:
    """Update the shipping address for a customer order. Pass order ID and new address as a single string."""
    return f"Shipping address updated: {order_and_address}"


@tool
def process_refund(order_id: str) -> str:
    """Issue a refund for a customer order."""
    return f"Refund of $49.99 issued for order {order_id}. Confirmation: REF-7721"


@tool
def cancel_account(customer_id: str) -> str:
    """Cancel a customer account."""
    return f"Account {customer_id} has been permanently deleted"


gated_tools = [
    TrustGatedTool(tool=lookup_order, required_capabilities=["lookup_order"], min_trust_score=0.5),       # read-only lookup — low risk
    TrustGatedTool(tool=check_inventory, required_capabilities=["check_inventory"], min_trust_score=0.5), # read-only query — low risk
    TrustGatedTool(tool=update_shipping, required_capabilities=["update_shipping"], min_trust_score=0.6), # modifies order — moderate risk
    TrustGatedTool(tool=cancel_account, required_capabilities=["cancel_account"], min_trust_score=0.95),  # destructive action — near-maximum trust required
    TrustGatedTool(tool=process_refund, required_capabilities=["process_refund"], min_trust_score=0.5),   # financial action — low trust floor, but bot lacks capability
]

print(f"Created {len(gated_tools)} trust-gated tools:")
for t in gated_tools:
    print(f"  {t.name:<20} requires {t.required_capabilities}, min trust {t.min_trust_score}")

## Step 4 — Configure policy, rate limiter, and executor

Two layers of governance:
1. **TokenBucket** — rate limiter that allows a burst of 5 tool calls, refilling 1 per second
2. **TrustedToolExecutor** — checks the chatbot's identity and capabilities before each tool call, and logs every invocation for auditing

In [ ]:
from langchain_agentmesh import TrustPolicy, TrustedToolExecutor
from agent_os.policies.rate_limiting import RateLimitConfig, TokenBucket

policy = TrustPolicy(
    require_verification=True,
    min_trust_score=0.7,
    audit_all_calls=True,
    cache_ttl_seconds=0,  # disable cache so each tool check evaluates capabilities fresh
)

rate_config = RateLimitConfig(capacity=5, refill_rate=1.0)
rate_bucket = TokenBucket.from_config(rate_config)

executor = TrustedToolExecutor(
    identity=bot_identity,
    policy=policy,
    tools=gated_tools,
)

print(f"Policy:     min_trust={policy.min_trust_score}, audit={policy.audit_all_calls}")
print(f"Rate limit: {rate_config.capacity} burst, {rate_config.refill_rate}/sec refill")
print(f"Executor:   {len(executor.list_tools())} tools — {executor.list_tools()}")

## Step 5 — Build the LangChain agent

Now we create a real LangChain agent using OpenAI. The agent decides **which** tool to call based on the customer's message. Our governance layer decides **whether** to allow it.

Each tool the agent sees is a governed wrapper — it looks like a normal tool, but internally checks the rate limiter and trust policy before executing.

```
Customer message
       │
       ▼
LangChain Agent (OpenAI decides which tool)
       │
       ▼
governed_tool_call()
  ├─ Rate limit check (TokenBucket)
  ├─ Capability check (TrustedToolExecutor)
  │
  ├─ PASS → execute tool, return result
  └─ FAIL → return error to agent
```

In [ ]:
from langchain.agents import create_agent


def governed_tool_call(tool_name: str, tool_input: str) -> str:
    """Run a tool through the governance layer (rate limit + trust check)."""
    # 1. Check rate limit
    if not rate_bucket.consume():
        wait = rate_bucket.time_until_available()
        print(f"  ⚡ {tool_name}({tool_input!r}) → RATE LIMITED; retry in ~{wait:.1f}s")  # demo visibility
        return "This action is temporarily unavailable due to high demand. Please try again shortly."

    # 2. Check trust and capabilities
    try:
        result = executor.invoke(tool_name, tool_input, invoker_card=bot_card)
        print(f"  ✅ {tool_name}({tool_input!r}) → allowed")  # demo visibility
        return result
    except PermissionError as e:
        print(f"  🚫 {tool_name}({tool_input!r}) → BLOCKED: {e}")  # demo visibility
        return "This action is not available. Please contact a human support agent for help."


@tool
def governed_lookup_order(order_id: str) -> str:
    """Look up a customer order by ID to check its status and shipping info."""
    return governed_tool_call("lookup_order", order_id)


@tool
def governed_check_inventory(product: str) -> str:
    """Check if a product is currently in stock."""
    return governed_tool_call("check_inventory", product)


@tool
def governed_update_shipping(order_and_address: str) -> str:
    """Update the shipping address for a customer order. Pass order ID and new address as a single string."""
    return governed_tool_call("update_shipping", order_and_address)


@tool
def governed_process_refund(order_id: str) -> str:
    """Issue a refund for a customer order."""
    return governed_tool_call("process_refund", order_id)


@tool
def governed_cancel_account(customer_id: str) -> str:
    """Cancel a customer account."""
    return governed_tool_call("cancel_account", customer_id)


agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[
        governed_lookup_order,
        governed_check_inventory,
        governed_update_shipping,
        governed_process_refund,
        governed_cancel_account,
    ],
    system_prompt=(
        "You are a customer support chatbot. Help customers with their orders, "
        "inventory questions, shipping updates, and account requests. "
        "IMPORTANT: Always call the appropriate tool for every customer request, "
        "including destructive actions like account cancellation or refunds. "
        "Never refuse a request yourself — always call the tool and let the "
        "system decide. If a tool says the action is not available, apologize "
        "and let the customer know that a human agent can help them with this "
        "request. Never mention internal systems, verification, trust, "
        "permissions, restrictions, or any technical reason for the failure."
    ),
)

print("LangChain agent ready with governed tools.")

## Step 6 — Test policy enforcement

Let's send five customer messages. Each one targets a different governance scenario. The LLM decides which tool to call, and our governance layers enforce the rules.

In [ ]:
customer_messages = [
    "Where is my order #12345?",
    "Is the blue backpack in stock?",
    "Update the shipping address on order #12345 to 456 Oak Ave, Springfield",
    "Delete my account, my customer ID is CUST-9182",
    "I want a refund for order #12345",
]

print("=" * 60)
print("  Customer Support Chat")
print("=" * 60)

for message in customer_messages:
    print(f"\nCustomer: {message}")
    result = agent.invoke(
        {"messages": [{"role": "user", "content": message}]}
    )
    final_message = result["messages"][-1]
    print(f"Chatbot:  {final_message.content}")

print("\n" + "=" * 60)

## Step 7 — Rate limiting in action

The rate limiter prevents any agent from making too many tool calls in a short time. Let's demonstrate by firing rapid requests that exhaust the token bucket.

In [ ]:
import time

# Reset the bucket so the demo starts from a known state
rate_bucket.reset()
print(f"Tokens available: {rate_bucket.available:.0f}/{int(rate_config.capacity)}")

# Phase 1: exhaust the bucket with rapid calls
print("Phase 1 — Rapid-fire calls (exhaust the bucket):")
for i in range(6):
    governed_tool_call("lookup_order", f"ORDER-{i}")

print(f"Tokens remaining: {rate_bucket.available:.0f}/{int(rate_config.capacity)}")

# Phase 2: wait and show recovery
print(f"Phase 2 — Wait 3 seconds (refill rate: {rate_config.refill_rate}/sec):")
time.sleep(3)
print(f"Tokens after 3s: {rate_bucket.available:.0f}/{int(rate_config.capacity)}")

# Phase 3: the bucket has refilled — calls work again
print("Phase 3 — Call again after recovery:")
governed_tool_call("lookup_order", "ORDER-99")
print(f"Tokens remaining: {rate_bucket.available:.0f}/{int(rate_config.capacity)}")

## Step 8 — View the audit trail

Every tool invocation that reaches the executor is recorded — both allowed and blocked calls. Rate-limited calls are caught before the executor, so they don't appear here; a production system would log those separately.

In [ ]:
audit_log = executor.get_audit_log()

print("=" * 60)
print("  Audit Trail")
print("=" * 60)

for i, record in enumerate(audit_log, 1):
    icon = "\u2705" if record.verified else "\U0001f6ab"
    print(f"\n  [{i}] {icon} {record.tool_name}")
    print(f"      Invoker DID: {record.invoker_did or 'self'}")
    print(f"      Verified:    {record.verified}")
    print(f"      Trust Score: {record.trust_score}")
    print(f"      Input:       {record.input_summary[:60]}")
    print(f"      Result:      {record.result_summary[:60]}")

allowed_count = sum(1 for r in audit_log if r.verified)
blocked_count = len(audit_log) - allowed_count

print("\n" + "-" * 60)
print(f"  Summary: {allowed_count} allowed, {blocked_count} blocked out of {len(audit_log)} tool calls")
print(f"  Rate limiter tokens remaining: {rate_bucket.available:.0f}/{int(rate_config.capacity)}")
print("=" * 60)

## What You Learned

- How to create a **cryptographic identity** for an agent using `VerificationIdentity` and `TrustedAgentCard`
- How to wrap LangChain tools with **trust gates** using `TrustGatedTool` — requiring specific capabilities and trust scores
- How **capability blocking** works — the agent tried to issue a refund but lacked the `process_refund` capability
- How **trust-score blocking** works — the agent had the `cancel_account` capability but its trust score (0.7) was below the required threshold (0.95)
- How **rate limiting** with `TokenBucket` prevents runaway tool usage — even allowed tools get throttled under load
- How to build a real **LangChain agent** (with OpenAI) that routes through a governance layer
- How to inspect the **audit trail** for compliance and debugging

**Previous:** [Multi-Agent Governance notebook](./03_multi_agent_governance.ipynb)

**Related tutorials:**
- [Trust & Identity](../docs/tutorials/02-trust-and-identity.md) — deeper dive into DIDs and trust scoring
- [Framework Integrations](../docs/tutorials/03-framework-integrations.md) — governing LangChain, CrewAI, AutoGen, and more